In [1]:
pip install google-generativeai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import  google.generativeai as googlegenai

c:\Users\Mohan Raj P\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Mohan Raj P\AppData\Local\Temp\ipykernel_36672\3393063020.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import  google.generativeai as googlegenai


In [3]:
import os 


In [4]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
os.getenv(".env")

In [6]:
from dotenv import load_dotenv
import json
from contextlib import AsyncExitStack
from agents import Runner, trace, add_trace_processor
from IPython.display import Markdown, display
from backend.market import get_share_price
from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.reset import reset_traders
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from backend.traders import get_researcher, get_researcher_tool, Trader
from backend.tracers import LogTracer

load_dotenv(override=True)

True

In [7]:
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [8]:
get_share_price("BMW")

Massive API unavailable (No Massive price available for BMW); using a simulated price


206.7

In [9]:
warren = Account.get("MOHAN")
print("Balance:", warren.balance)
print("Strategy:", warren.get_strategy())

Balance: 9120.4444
Strategy: 


In [10]:
servers = trader_mcp_servers() + researcher_mcp_servers("MOHAN")
count = 0
for server in servers:
    async with server:
        tools = await server.list_tools()
        count += len(tools)
print(f"We have {len(servers)} MCP servers, and {count} tools")

We have 6 MCP servers, and 17 tools


In [11]:
async with AsyncExitStack() as stack:
    servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("MOHAN")]
    researcher = await get_researcher(servers, "gemini-2.5-flash")
    with trace("Researcher"):
        result = await Runner.run(researcher, "What's the latest news on Amazon?", max_turns=30)
display(Markdown(result.final_output))

Here's a summary of the latest news on Amazon from the past week:

Amazon is making significant strides in AI, with the introduction of "Evry" smart delivery glasses that act as a wearable AI assistant for delivery drivers. They also launched Claude Sonnet 5 on Amazon Bedrock and continue to develop their Zoox robotaxi.

However, despite these investments, Amazon has also confirmed a new round of layoffs within its Artificial General Intelligence (AGI) organization. This is part of an ongoing restructuring effort to sharpen its focus on key initiatives, though the exact number of affected employees is unspecified by Amazon, some reports suggest around 16,000 job cuts.

In [12]:
researcher_tool = await get_researcher_tool(researcher_mcp_servers("MOHAN"), "gpt-4o-mini")
print("Tool name:", researcher_tool.name)
print("Description:", researcher_tool.description)

Tool name: Researcher
Description: This tool researches online for news and opportunities, either based on your specific request to look into a certain stock, or generally for notable financial news and opportunities. Describe what kind of research you're looking for.


In [13]:
add_trace_processor(LogTracer())
warren = Trader("MOHAN", "Patience", "gpt-5.4-mini")
await warren.run()

Error getting response: Error code: 429 - {'error': {'message': 'Request too large for gpt-5.4-mini in organization org-wTLboYmqHSfUYGVeYoWbGz2r on tokens per min (TPM): Limit 200000, Requested 217983. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}. (request_id: 828810ac-d22e-43fa-ba7b-319a46bf4dcc)


In [14]:
resource = await read_accounts_resource("warren")
info = json.loads(resource)
print(info["transactions"][-1] if info["transactions"] else "No transactions found")

{'symbol': 'MTUM', 'quantity': 10, 'price': 228.92694, 'timestamp': '2026-07-25 15:37:28', 'rationale': 'Core momentum exposure with diversification and liquidity; fits risk-controlled growth strategy and preserves cash buffer while expressing broad market trend.'}
